In [2]:
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.utils import class_weight
from sklearn.metrics import accuracy_score, precision_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, ELU
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
import optuna
import json

# Paths
base_path = './Competition_data'
dataset_folders = os.listdir(base_path)

In [3]:
# Define a function for hyperparameter tuning using Optuna
def tune_nn_model(trial, input_dim):
    model = Sequential()
    # Suggest number of units for each layer
    n_units_1 = trial.suggest_int('n_units_1', 64, 256)
    n_units_2 = trial.suggest_int('n_units_2', 32, 128)
    n_units_3 = trial.suggest_int('n_units_3', 16, 64)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.3)
    l2_reg = trial.suggest_float('l2_reg', 1e-5, 1e-3)

    model.add(Dense(n_units_1, input_dim=input_dim, activation='elu', kernel_regularizer=l2(l2_reg)))
    model.add(BatchNormalization())
    model.add(Dropout(rate=dropout_rate))
    
    model.add(Dense(n_units_2, activation='elu', kernel_regularizer=l2(l2_reg)))
    model.add(BatchNormalization())
    model.add(Dropout(rate=dropout_rate))

    model.add(Dense(n_units_3, activation='elu', kernel_regularizer=l2(l2_reg)))
    model.add(BatchNormalization())
    model.add(Dropout(rate=dropout_rate))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# Initialize lists to store summary metrics for all datasets
summary_metrics = []

# Define callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.00001)

# Loop through each dataset folder
for dataset in dataset_folders:
    dataset_path = os.path.join(base_path, dataset)

    # Load data
    X_train = pd.read_csv(os.path.join(dataset_path, 'X_train.csv'))
    y_train = pd.read_csv(os.path.join(dataset_path, 'y_train.csv'))
    X_test = pd.read_csv(os.path.join(dataset_path, 'X_test.csv'))

    # Ensure y_train is a one-dimensional array
    if len(y_train.shape) > 1 and y_train.shape[1] == 1:
        y_train = y_train.squeeze()

    # Data Preprocessing
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Handle Class Imbalance using SMOTE
    smote = SMOTE(random_state=42)
    X_train_scaled, y_train = smote.fit_resample(X_train_scaled, y_train)

    # Perform Optuna Hyperparameter Tuning once per dataset
    def objective(trial):
        model = Sequential()
        model.add(Dense(trial.suggest_int('n_units_1', 32, 128),
                        input_dim=X_train.shape[1],
                        activation='elu',
                        kernel_regularizer=l2(trial.suggest_float('l2_reg', 1e-5, 1e-2, log=True))))
        model.add(BatchNormalization())
        model.add(Dropout(rate=trial.suggest_float('dropout_rate', 0.1, 0.5)))
        model.add(Dense(trial.suggest_int('n_units_2', 32, 128), activation='elu'))
        model.add(BatchNormalization())
        model.add(Dropout(rate=trial.suggest_float('dropout_rate', 0.1, 0.5)))
        model.add(Dense(1, activation='sigmoid'))
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])

        history = model.fit(
            X_train_scaled, y_train,
            validation_split=0.2,
            epochs=20,
            batch_size=16,
            verbose=0
        )
        val_auc = max(history.history['val_auc'])
        return val_auc

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=10)

    best_params = study.best_params
    print(f"Best params for {dataset}: {best_params}")

    # Train Neural Network using best parameters
    nn_model = Sequential()
    nn_model.add(Dense(best_params['n_units_1'], input_dim=X_train.shape[1],
                       activation='elu', kernel_regularizer=l2(best_params['l2_reg'])))
    nn_model.add(BatchNormalization())
    nn_model.add(Dropout(rate=best_params['dropout_rate']))
    nn_model.add(Dense(best_params['n_units_2'], activation='elu'))
    nn_model.add(BatchNormalization())
    nn_model.add(Dropout(rate=best_params['dropout_rate']))
    nn_model.add(Dense(1, activation='sigmoid'))
    nn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])

    # Cross-Validation using KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    val_aucs = []
    meta_features_train = []
    meta_labels_train = []

    for train_index, val_index in kf.split(X_train_scaled):
        X_train_split, X_val = X_train_scaled[train_index], X_train_scaled[val_index]
        y_train_split, y_val = y_train.iloc[train_index], y_train.iloc[val_index]

        # Train Neural Network
        history = nn_model.fit(
            X_train_split, y_train_split,
            validation_data=(X_val, y_val),
            epochs=50, batch_size=16, verbose=0,
            callbacks=[early_stopping, reduce_lr]
        )
        val_auc = max(history.history['val_auc'])
        val_aucs.append(val_auc)

        # Train Random Forest
        rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
        rf_model.fit(X_train_split, y_train_split)

        # Train XGBoost
        xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='logloss')
        xgb_model.fit(X_train_split, y_train_split)

        # Gather Meta Features for Meta-Model
        nn_val_pred = nn_model.predict(X_val).flatten()
        rf_val_pred = rf_model.predict_proba(X_val)[:, 1]
        xgb_val_pred = xgb_model.predict_proba(X_val)[:, 1]

        meta_features = np.vstack((nn_val_pred, rf_val_pred, xgb_val_pred)).T
        meta_features_train.append(meta_features)
        meta_labels_train.append(y_val)

    # Compute and save average validation AUC
    avg_val_auc = np.mean(val_aucs)
    summary_metrics.append({'Dataset': dataset, 'Average Validation AUC': avg_val_auc})
    print(f"Dataset: {dataset}, Average Validation AUC: {avg_val_auc}")

    # Train Meta-Model (Logistic Regression)
    meta_features_train = np.vstack(meta_features_train)
    meta_labels_train = np.hstack(meta_labels_train)
    meta_model = LogisticRegression(random_state=42)
    meta_model.fit(meta_features_train, meta_labels_train)

    # Prepare Meta Features for Test Set
    nn_test_pred = nn_model.predict(X_test_scaled).flatten()
    rf_test_pred = rf_model.predict_proba(X_test_scaled)[:, 1]
    xgb_test_pred = xgb_model.predict_proba(X_test_scaled)[:, 1]
    meta_features_test = np.vstack((nn_test_pred, rf_test_pred, xgb_test_pred)).T
    y_test_pred_percentage = meta_model.predict_proba(meta_features_test)[:, 1]

    # Save Predictions
    y_predict_path = os.path.join(dataset_path, 'y_predict.csv')
    pd.DataFrame({'Prediction': y_test_pred_percentage}).to_csv(y_predict_path, index=False)

    # Save best parameters
    best_params_path = os.path.join(dataset_path, 'best_params.json')
    with open(best_params_path, 'w') as f:
        json.dump(best_params, f)

# Save summary metrics
summary_path = os.path.join(base_path, 'summary_metrics.json')
with open(summary_path, 'w') as f:
    json.dump(summary_metrics, f)


In [5]:
# 定義資料夾路徑
base_path = "./Competition_data"  # 替換成你的路徑
dataset_folders = os.listdir(base_path)

# 初始化存放所有資料集最佳參數的字典
all_best_params = {}

# 逐一處理每個資料夾
for dataset in dataset_folders:
    dataset_path = os.path.join(base_path, dataset)
    best_params_path = os.path.join(dataset_path, 'best_params.json')
    
    # 檢查是否存在最佳參數檔案
    if os.path.exists(best_params_path):
        with open(best_params_path, 'r') as f:
            best_params = json.load(f)
        
        # 加入到結果字典，標註資料集名稱
        all_best_params[dataset] = best_params

# 將整合結果存成單一檔案
output_path = os.path.join(base_path, 'all_best_params.json')
with open(output_path, 'w') as f:
    json.dump(all_best_params, f, indent=4)

print(f"所有最佳參數已整合完成，存於: {output_path}")

所有最佳參數已整合完成，存於: ./Competition_data\all_best_params.json


In [6]:
import json
import pandas as pd

# 定義檔案路徑
all_best_params_path = "all_best_params.json"
summary_metrics_path = "summary_metrics.json"

# 讀取 JSON 檔案
with open(all_best_params_path, 'r') as f:
    all_best_params = json.load(f)

with open(summary_metrics_path, 'r') as f:
    summary_metrics = json.load(f)

# 按資料集名稱的數字部分排序
def extract_number(dataset_name):
    return int(dataset_name.split('_')[1])

all_best_params_ordered = dict(sorted(all_best_params.items(), key=lambda x: extract_number(x[0])))
summary_metrics_ordered = sorted(summary_metrics, key=lambda x: extract_number(x['Dataset']))

# 將排序後的資料轉成 DataFrame
best_params_df = pd.DataFrame.from_dict(all_best_params_ordered, orient='index').reset_index()
best_params_df.rename(columns={'index': 'Dataset'}, inplace=True)

summary_metrics_df = pd.DataFrame(summary_metrics_ordered)

# 將結果存成 CSV 檔案
best_params_csv_path = "all_best_params_ordered.csv"
summary_metrics_csv_path = "summary_metrics_ordered.csv"

best_params_df.to_csv(best_params_csv_path, index=False)
summary_metrics_df.to_csv(summary_metrics_csv_path, index=False)

print(f"所有最佳參數已儲存至: {best_params_csv_path}")
print(f"摘要指標已儲存至: {summary_metrics_csv_path}")


所有最佳參數已儲存至: all_best_params_ordered.csv
摘要指標已儲存至: summary_metrics_ordered.csv
